# Tennis Serve Biomechanics Analysis
### AI-Powered Pose Estimation with MediaPipe BlazePose

This notebook presents a data-driven biomechanical analysis of the tennis serve.  
Using **MediaPipe BlazePose**, joint angles and wrist kinematics are extracted frame-by-frame  
from match and practice footage of two professional players — **Novak Djokovic** and **Roger Federer** —  
and one **amateur** player.

**Objective:** Identify biomechanical differences between elite and recreational serve mechanics  
to derive actionable coaching insights.

---

## Table of Contents
1. [Setup & Imports](#1.-Setup-&-Imports)
2. [Data Loading & Exploration](#2.-Data-Loading-&-Exploration)
3. [Single-Serve Deep Dive: Impact Detection](#3.-Single-Serve-Deep-Dive:-Impact-Detection)
4. [Professional Comparison: Djokovic vs Federer](#4.-Professional-Comparison:-Djokovic-vs-Federer)
5. [Three-Player Comparison](#5.-Three-Player-Comparison)
6. [Key Findings & Conclusions](#6.-Key-Findings-&-Conclusions)

---
## 1. Setup & Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

warnings.filterwarnings("ignore")

# Global plot style
plt.rcParams.update({
    "figure.dpi":        120,
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
})

COLORS = {"Djokovic": "#1565C0", "Federer": "#2E7D32", "Amateur": "#E65100"}

print("Environment ready.")

---
## 2. Data Loading & Exploration

Three video clips were analysed using MediaPipe BlazePose (model complexity = 1).  
Each row in the output CSVs represents one video frame with per-frame pose metrics.

| Feature | Joint Landmarks | Biomechanical Meaning |
|---|---|---|
| `elbow_angle` | Shoulder → Elbow → Wrist | Arm extension at contact |
| `knee_angle` | Hip → Knee → Ankle | Leg drive and ground force |
| `shoulder_angle` | Hip → Shoulder → Elbow | Arm abduction / elevation |
| `wrist_height` | Wrist Y (normalised 0–1) | Toss height (lower = physically higher) |

In [ ]:
djokovic = pd.read_csv("../outputs/djokovic_analysis.csv")
federer  = pd.read_csv("../outputs/federer_analysis.csv")
amateur  = pd.read_csv("../outputs/amateur_analysis.csv")

for name, df in [("Djokovic", djokovic), ("Federer", federer), ("Amateur", amateur)]:
    print(f"{name:<10}: {len(df):>5} frames | "
          f"frame range [{df['frame'].min():.0f} – {df['frame'].max():.0f}]")

print("\nSample — Federer:")
federer.head()

In [ ]:
# Overview: raw biomechanics across the Federer serve segment
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Federer — Full Serve Segment Biomechanics", fontsize=14, fontweight="bold")

metrics = [
    ("elbow_angle",    "Elbow Angle (°)"),
    ("knee_angle",     "Knee Angle (°)"),
    ("shoulder_angle", "Shoulder Angle (°)"),
    ("wrist_height",   "Wrist Height (norm.)"),
]
for ax, (col, label) in zip(axes.flatten(), metrics):
    ax.plot(federer["frame"], federer[col], color=COLORS["Federer"], linewidth=1.5)
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("Frame")
    ax.set_ylabel(label)
    ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

---
## 3. Single-Serve Deep Dive: Impact Detection

**Goal:** Identify the exact frame of racket–ball contact within the serve motion.

**Method — Biomechanical Impact Score:**  
At the true impact frame, the arm is near-fully extended (high elbow angle),  
the legs are pushing upward (high knee angle), and the wrist is at its highest point  
(low wrist Y-value in image coordinates). We operationalise this as:

$$\text{score} = \text{elbow\_angle} + \text{knee\_angle} - 300 \times \text{wrist\_height}$$

The frame maximising this score is our predicted impact frame.  
The coefficient 300 was calibrated empirically so the argmax aligns with the visually identifiable contact frame.

In [ ]:
# Compute biomechanical impact score and find the best frame
federer_serve = federer.copy()
federer_serve["score"] = (
    federer_serve["elbow_angle"] +
    federer_serve["knee_angle"] -
    300 * federer_serve["wrist_height"]
)

best_row         = federer_serve.loc[federer_serve["score"].idxmax()]
impact_frame_num = int(best_row["frame"])

print(f"Detected impact frame  : {impact_frame_num}")
print(f"Elbow angle            : {best_row['elbow_angle']:.1f}°")
print(f"Knee angle             : {best_row['knee_angle']:.1f}°")
print(f"Wrist height           : {best_row['wrist_height']:.4f}")
print(f"Composite impact score : {best_row['score']:.1f}")

In [ ]:
# Dual-axis plot: joint angles (left) + normalised wrist height (right)
wh = federer_serve["wrist_height"]
federer_serve["wrist_norm"] = (wh - wh.min()) / (wh.max() - wh.min())

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

angle_series = [
    ("elbow_angle",    "Elbow",    "#1565C0"),
    ("knee_angle",     "Knee",     "#2E7D32"),
    ("shoulder_angle", "Shoulder", "#6A1B9A"),
]
for col, label, color in angle_series:
    ax1.plot(federer_serve["frame"], federer_serve[col],
             label=label, color=color, linewidth=1.8)

# Wrist height on inverted secondary axis (lower Y = wrist physically higher)
ax2.plot(federer_serve["frame"], federer_serve["wrist_norm"],
         label="Wrist Height (norm.)", color="#E65100",
         linewidth=1.8, linestyle="--")
ax2.invert_yaxis()
ax2.set_ylabel("Normalised Wrist Height (inverted)", fontsize=10, color="#E65100")
ax2.tick_params(axis="y", colors="#E65100")

# Mark impact
ax1.axvline(impact_frame_num, color="red", linestyle=":", linewidth=2.5, label="Impact")
ax1.set_xlabel("Frame")
ax1.set_ylabel("Joint Angle (°)")
ax1.set_title(
    f"Federer — Serve Biomechanics with Detected Impact (frame {impact_frame_num})",
    fontsize=13, fontweight="bold"
)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right")
ax1.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Display pre-impact, impact, and follow-through frames from the video
VIDEO_PATH = "../data/raw_videos/test_video.mp4"

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Federer — Serve Frame Sequence Around Impact", fontsize=13, fontweight="bold")

offsets = [-3, 0, 3]
titles  = [
    f"Frame {impact_frame_num - 3}\nPre-impact",
    f"Frame {impact_frame_num}\n★ IMPACT",
    f"Frame {impact_frame_num + 3}\nFollow-through",
]

for ax, offset, title in zip(axes, offsets, titles):
    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, impact_frame_num + offset)
    ret, frame = cap.read()
    cap.release()
    if ret:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontweight="bold" if offset == 0 else "normal")
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## 4. Professional Comparison: Djokovic vs Federer

To compare the two professionals on a common timeline, we:
1. Detect each player's impact frame using the biomechanical score method.
2. Extract a ±40-frame window (~1.3 s at 30 fps) around each impact.
3. Set **relative frame 0 = impact** so both serve motions are aligned.
4. Apply a 3-frame centred rolling mean to smooth MediaPipe estimation noise.

This alignment allows direct frame-by-frame comparison of the serve mechanics  
independent of when in the video each serve occurs.

In [ ]:
def detect_impact(df):
    """Return the row with the highest biomechanical impact score."""
    scored = df.copy()
    scored["score"] = (
        scored["elbow_angle"] +
        scored["knee_angle"] -
        300 * scored["wrist_height"]
    )
    return scored.loc[scored["score"].idxmax()]


def extract_window(df, impact_frame, window=40, smooth_cols=None):
    """Extract ±window frames around impact, add relative_frame, optionally smooth."""
    win = df[
        (df["frame"] >= impact_frame - window) &
        (df["frame"] <= impact_frame + window)
    ].copy()
    win["relative_frame"] = win["frame"] - impact_frame
    if smooth_cols:
        for col in smooth_cols:
            win[f"{col}_smooth"] = win[col].rolling(3, center=True, min_periods=1).mean()
    return win


# Detect impact and extract windows for both professionals
djokovic_impact = detect_impact(djokovic)
djokovic_win    = extract_window(
    djokovic, int(djokovic_impact["frame"]),
    smooth_cols=["elbow_angle", "shoulder_angle"]
)

federer_impact = detect_impact(federer)
federer_win    = extract_window(
    federer, int(federer_impact["frame"]),
    smooth_cols=["elbow_angle", "shoulder_angle"]
)

print(f"{'Player':<12} {'Impact Frame':>14} {'Elbow':>8} {'Knee':>8} {'Shoulder':>10}")
print("-" * 55)
for name, row in [("Djokovic", djokovic_impact), ("Federer", federer_impact)]:
    print(f"{name:<12} {int(row['frame']):>14} "
          f"{row['elbow_angle']:>7.1f}° "
          f"{row['knee_angle']:>7.1f}° "
          f"{row['shoulder_angle']:>9.1f}°")

In [ ]:
# Side-by-side: elbow, knee, and shoulder angles aligned to impact
metrics_plot = [
    ("elbow_angle_smooth",    "Elbow Angle (°)"),
    ("knee_angle",            "Knee Angle (°)"),
    ("shoulder_angle_smooth", "Shoulder Angle (°)"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    "Djokovic vs Federer — Joint Angles Aligned to Impact (frame 0)",
    fontsize=14, fontweight="bold"
)

for ax, (col, label) in zip(axes, metrics_plot):
    for name, win in [("Djokovic", djokovic_win), ("Federer", federer_win)]:
        ax.plot(win["relative_frame"], win[col],
                color=COLORS[name], linewidth=2.2, label=name)
    ax.axvline(0, color="red", linestyle="--", linewidth=1.5, alpha=0.7, label="Impact")
    ax.set_xlabel("Frames Relative to Impact")
    ax.set_ylabel(label)
    ax.set_title(label, fontweight="bold")
    ax.legend()
    ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Angular velocity = frame-over-frame derivative of the smoothed angle
for win in [djokovic_win, federer_win]:
    win["elbow_velocity"]    = win["elbow_angle_smooth"].diff()
    win["shoulder_velocity"] = win["shoulder_angle_smooth"].diff()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Djokovic vs Federer — Angular Velocity Around Impact",
    fontsize=13, fontweight="bold"
)

vel_cfg = [
    ("elbow_velocity",    "Elbow Angular Velocity (°/frame)"),
    ("shoulder_velocity", "Shoulder Angular Velocity (°/frame)"),
]
for ax, (col, label) in zip(axes, vel_cfg):
    for name, win in [("Djokovic", djokovic_win), ("Federer", federer_win)]:
        ax.plot(win["relative_frame"], win[col],
                color=COLORS[name], linewidth=2, label=name)
    ax.axvline(0, color="red",   linestyle="--", linewidth=1.5, alpha=0.7, label="Impact")
    ax.axhline(0, color="black", linewidth=0.8,  alpha=0.4)
    ax.set_xlabel("Frames Relative to Impact")
    ax.set_ylabel(label)
    ax.set_title(label, fontweight="bold")
    ax.legend()
    ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Structured comparison table at impact
comparison = pd.DataFrame({
    "Metric": [
        "Elbow Angle at Impact (°)",
        "Knee Angle at Impact (°)",
        "Shoulder Angle at Impact (°)",
        "Wrist Height at Impact",
    ],
    "Djokovic": [
        f"{djokovic_impact['elbow_angle']:.1f}",
        f"{djokovic_impact['knee_angle']:.1f}",
        f"{djokovic_impact['shoulder_angle']:.1f}",
        f"{djokovic_impact['wrist_height']:.3f}",
    ],
    "Federer": [
        f"{federer_impact['elbow_angle']:.1f}",
        f"{federer_impact['knee_angle']:.1f}",
        f"{federer_impact['shoulder_angle']:.1f}",
        f"{federer_impact['wrist_height']:.3f}",
    ],
})
comparison.set_index("Metric")

---
## 5. Three-Player Comparison

We now add the **amateur** player and repeat the same analysis.  
The same impact detection method and window extraction are applied unchanged,  
ensuring a fair apples-to-apples comparison.

**Research question:** Where does the amateur's serve mechanics deviate from the professionals?

In [ ]:
# Detect impact and extract window for the amateur
amateur_impact = detect_impact(amateur)
amateur_win    = extract_window(
    amateur, int(amateur_impact["frame"]),
    smooth_cols=["elbow_angle", "shoulder_angle"]
)

print(f"Amateur impact frame   : {int(amateur_impact['frame'])}")
print(f"Elbow angle            : {amateur_impact['elbow_angle']:.1f}°")
print(f"Knee angle             : {amateur_impact['knee_angle']:.1f}°")
print(f"Shoulder angle         : {amateur_impact['shoulder_angle']:.1f}°")
print(f"Wrist height           : {amateur_impact['wrist_height']:.3f}")

In [ ]:
# 2x2 panel: all four biomechanical signals for all three players
all_players = [
    ("Djokovic", djokovic_win),
    ("Federer",  federer_win),
    ("Amateur",  amateur_win),
]

plot_cfg = [
    ("elbow_angle_smooth",    "Elbow Angle (°)"),
    ("knee_angle",            "Knee Angle (°)"),
    ("shoulder_angle_smooth", "Shoulder Angle (°)"),
    ("wrist_height",          "Wrist Height (norm.)"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "Three-Player Biomechanical Comparison — Aligned to Impact (frame 0)",
    fontsize=14, fontweight="bold"
)

for ax, (col, label) in zip(axes.flatten(), plot_cfg):
    for name, win in all_players:
        ax.plot(win["relative_frame"], win[col],
                color=COLORS[name], linewidth=2.2, label=name)
    ax.axvline(0, color="red", linestyle="--", linewidth=1.5, alpha=0.7, label="Impact")
    ax.set_xlabel("Frames Relative to Impact")
    ax.set_ylabel(label)
    ax.set_title(label, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Wrist velocity comparison on normalised time (0 = window start, 1 = end)
def norm_time_velocity(win):
    df = win.copy()
    lo, hi = df["relative_frame"].min(), df["relative_frame"].max()
    df["time_norm"]              = (df["relative_frame"] - lo) / (hi - lo)
    df["wrist_velocity"]         = df["wrist_height"].diff()
    df["wrist_velocity_smooth"]  = df["wrist_velocity"].rolling(3, center=True, min_periods=1).mean()
    return df

dj_norm  = norm_time_velocity(djokovic_win)
fed_norm = norm_time_velocity(federer_win)
am_norm  = norm_time_velocity(amateur_win)

fig, ax = plt.subplots(figsize=(12, 5))

for name, df in [("Djokovic", dj_norm), ("Federer", fed_norm), ("Amateur", am_norm)]:
    ax.plot(df["time_norm"], df["wrist_velocity_smooth"],
            color=COLORS[name], linewidth=2.2, label=name)

ax.axvline(0.5, color="red",   linestyle="--", linewidth=1.5, alpha=0.7, label="Impact (~50%)")
ax.axhline(0,   color="black", linewidth=0.8,  alpha=0.4)
ax.set_xlabel("Normalised Time  (0 = window start, 1 = window end)")
ax.set_ylabel("Smoothed Wrist Velocity (Δheight / frame)")
ax.set_title("Wrist Velocity Comparison — Normalised Serve Window",
             fontsize=13, fontweight="bold")
ax.legend()
ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar chart — peak angle at impact for all three players
players = ["Djokovic", "Federer", "Amateur"]
impacts = {"Djokovic": djokovic_impact, "Federer": federer_impact, "Amateur": amateur_impact}

metrics_bar = [
    ("Elbow Angle (°)",    "elbow_angle"),
    ("Knee Angle (°)",     "knee_angle"),
    ("Shoulder Angle (°)", "shoulder_angle"),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Peak Joint Angles at Impact — Three-Player Comparison",
             fontsize=14, fontweight="bold")

for ax, (label, key) in zip(axes, metrics_bar):
    vals = [impacts[p][key] for p in players]
    bars = ax.bar(players, vals,
                  color=[COLORS[p] for p in players], alpha=0.85, width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1,
                f"{val:.1f}°",
                ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_title(label, fontweight="bold")
    ax.set_ylabel(label)
    ax.set_ylim(0, 200)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics table
summary = pd.DataFrame({
    "Player": players,
    "Impact Frame":       [int(impacts[p]["frame"])            for p in players],
    "Elbow Angle (°)":    [round(impacts[p]["elbow_angle"], 1)    for p in players],
    "Knee Angle (°)":     [round(impacts[p]["knee_angle"], 1)     for p in players],
    "Shoulder Angle (°)": [round(impacts[p]["shoulder_angle"], 1) for p in players],
    "Wrist Height":       [round(impacts[p]["wrist_height"], 3)   for p in players],
})
summary.set_index("Player")

---
## 6. Key Findings & Conclusions

### Summary at Impact

| Metric | Djokovic | Federer | Amateur | Key Insight |
|---|---|---|---|---|
| **Elbow Angle** | ~180° | ~180° | ~157° | Amateur has ~23° less extension — direct power loss |
| **Knee Angle** | ~179° | ~173° | ~164° | All extend legs; professionals more explosively |
| **Shoulder Angle** | ~18° | ~149° | low | Federer's trunk-rotation dominant technique |
| **Wrist Height** | 0.59 | 0.46 | higher | Professionals achieve lower wrist Y (wrist physically higher) |

---

### Finding 1 — Elbow extension is a near-invariant of elite serving
Both professionals reach near-180° elbow angle at impact regardless of serve type or style.  
The amateur's ~157° deficit (~23°) is the single most actionable coaching cue — a bent arm  
at contact directly reduces racket-head speed and therefore serve power.

### Finding 2 — Shoulder angle is the primary professional differentiator
Federer's significantly larger shoulder abduction at impact reflects a **trunk-rotation dominant**  
technique, where power is generated through hip-to-shoulder rotation.  
Djokovic's lower shoulder angle indicates a more upright, **arm-dominant** motion with  
exceptional elbow extension consistency.

### Finding 3 — Wrist trajectory reveals toss quality
Professionals achieve higher wrist elevation (lower Y in image coordinates) earlier in the  
serve motion. The amateur's wrist rises more slowly and peaks lower, consistent with a  
less consistent or shorter ball toss.

### Finding 4 — Angular velocity shows the kinematic chain
The elbow velocity curve shows a sharp positive spike just before impact for both professionals  
(arm extending rapidly), followed by deceleration at contact.  
This whip-like acceleration pattern is the biomechanical signature of an efficient serve.

---

### Coaching Recommendations for the Amateur (Priority Order)

| Priority | Area | Intervention |
|---|---|---|
| 1 | Elbow extension | Shadow swings focusing on full arm straightening at the top |
| 2 | Shoulder rotation | Hip-to-shoulder rotation drills (medicine ball, shadow serve) |
| 3 | Toss consistency | Isolated toss practice — catch the ball at the contact point |
| 4 | Leg drive | Jump-serve drills to synchronise knee extension with arm contact |

---

### Limitations & Next Steps

- **2D only:** All angles are computed from a single camera; true 3D joint angles require multi-view or depth sensing.
- **Small amateur sample:** Conclusions about the amateur are directional, not statistically definitive.
- **Heuristic impact detection:** Ball tracking would provide a ground-truth impact reference.
- **Future work:** Serve type classification (flat/slice/kick), larger amateur cohort, automated LLM coaching feedback.